# AML detection: Azure ML SDK v2 pipeline

Run on the Azure ML kernel **Python 3.10 - SDK v2** (compute instance) or any Python >= 3.10 with `azure-ai-ml` and `azure-identity`.
The pipeline steps run in their own Python 3.11 job environment, so the kernel version does not matter for training.

Creates in your workspace: a **data asset**, an **environment**, five **components**, one **pipeline job** (prepare > features > imbalance > tune > final), and a registered **model**; every step logs parameters, metrics and PNG/JSON artifacts with MLflow.

Cost: the pipeline job uses your compute cluster (billable while nodes run). Set `DRY_RUN = True` to build everything offline without calling Azure.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from mlops.azure import build_job_code, pipeline_lib

DRY_RUN = False          # True: offline build only, no Azure calls
COMPUTE = "aml-mlops-compute"
UPLOAD_DATA = True       # False: reuse the latest saml-d-raw data asset
RAW_CSV = ROOT / "data" / "raw" / "SAML-D.csv"

## 1. Connect to the workspace (uses config.json or AZURE_* environment variables)

In [ ]:
if DRY_RUN:
    ml_client = None
else:
    ml_client = pipeline_lib.get_client()
    print("Workspace:", ml_client.workspace_name)

## 2. Compute target

In [ ]:
if not DRY_RUN:
    print(json.dumps(pipeline_lib.check_compute(ml_client, COMPUTE), indent=2))
    # If this is a compute instance and the job is rejected, create a scale-to-zero cluster:
    # COMPUTE = "aml-cluster"; pipeline_lib.create_cluster(ml_client, COMPUTE)

## 3. Data asset (raw SAML-D CSV, about 1 GB upload)

In [ ]:
if DRY_RUN:
    data_ref = f"azureml:{pipeline_lib.DATA_NAME}@latest"
else:
    if UPLOAD_DATA:
        data = pipeline_lib.register_data(ml_client, RAW_CSV)
    else:
        data = ml_client.data.get(pipeline_lib.DATA_NAME, label="latest")
    data_ref = f"azureml:{data.name}:{data.version}"
print(data_ref)

## 4. Environment and components

In [ ]:
build_job_code.build()
if DRY_RUN:
    components = pipeline_lib.load_components()
else:
    env = pipeline_lib.register_environment(ml_client)
    print("Environment:", env.name, env.version)
    components = pipeline_lib.register_components(ml_client)
print({k: f"{v.name}:{v.version}" for k, v in components.items()})

## 5. Pipeline job

In [ ]:
pipeline_job = pipeline_lib.build_pipeline(
    components, COMPUTE, data_ref, neg_pool=200_000, cv_neg=120_000
)
print("Steps:", list(pipeline_job.jobs))

In [ ]:
if not DRY_RUN:
    job = pipeline_lib.submit(ml_client, pipeline_job)
    print(job.name)
    print(job.studio_url)

## 6. Wait for completion (streams logs)

In [ ]:
if not DRY_RUN:
    status = pipeline_lib.wait(ml_client, job)
    print("Status:", status)

## 7. Register the model (versioned, with metadata tags)

In [ ]:
if not DRY_RUN and status == "Completed":
    model = pipeline_lib.register_model(ml_client, job.name)
    print("Registered:", model.name, model.version, model.tags)

## 8. Download every step's outputs and show the PNG charts

In [ ]:
if not DRY_RUN and status == "Completed":
    from IPython.display import Image, display

    folders = pipeline_lib.download_outputs(ml_client, job.name, "azure_outputs")
    for png in sorted(Path("azure_outputs").rglob("*.png")):
        print(png)
        display(Image(filename=str(png)))

## 9. Compare the logged metrics (MLflow, workspace tracking)

In [ ]:
if not DRY_RUN and status == "Completed":
    import mlflow

    mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)
    runs = mlflow.search_runs(experiment_names=[pipeline_lib.EXPERIMENT])
    cols = [c for c in runs.columns if c.startswith("metrics.test.pr_auc") or c == "tags.pipeline_step"]
    print(runs[cols].dropna(how="all", subset=cols[:-1]) if cols else runs.head())